# ODIR-5K Phase 1 — Siamese Nhị Phân Song Nhãn (Normal vs Pathological)

**Đồ án tốt nghiệp** · Phân loại nhị phân bằng mạng Siamese ghép cặp 2 mắt + hồi quy tuổi võng mạc (phụ trợ).
So sánh **EfficientNet-B0 (CNN)** vs **Swin-Tiny (Transformer)** qua 6 thực nghiệm ablation.

**Chuẩn bị trên Kaggle:**
1. Tạo Dataset code tên `odir5k-code` (chứa `src/`, `configs/`, `splits_clean/`, `train.py`, `evaluate.py`) — dùng `kaggle_upload/upload_code.sh`.
2. Add Dataset ảnh gốc **ODIR-5K** (thư mục chứa `Training Images`).
3. Bật **GPU T4/P100** · Internet **ON** (để timm tải pretrained ImageNet).
4. Run All.

> Ảnh enhanced (ROI Crop + Ben Graham + CLAHE) được **sinh tự động** tại runtime từ ảnh gốc — không cần upload.


In [ ]:
# CELL 1 — Cài thư viện
!pip install -q timm albumentations pyyaml scikit-learn tqdm
print('✅ Đã cài thư viện')

In [ ]:
# CELL 2 — Kiểm tra môi trường & TƯƠNG THÍCH GPU (fail-fast)
# Mục đích: phát hiện GPU không tương thích NGAY tại đây (vd Tesla P100 / sm_60),
#           tránh lãng phí ~25 phút tiền xử lý ở CELL 5 rồi mới crash khi vào training.
# Lý do: PyTorch >= 2.10 đã bỏ hỗ trợ kiến trúc Pascal (P100, sm_60).
#        Cần GPU T4 (sm_75) trở lên — đổi tại Session options (⚙️) → Accelerator → GPU T4 x2.
import os, sys, torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

if not torch.cuda.is_available():
    raise SystemExit('❌ Không thấy GPU. Bật Session options (⚙️) → Accelerator → GPU T4 x2 → Save.')

name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
cc = major * 10 + minor
supported = torch.cuda.get_arch_list()  # vd: ['sm_70', 'sm_75', 'sm_80', ...]
print(f'GPU: {name} | compute capability: sm_{cc}')
print('PyTorch hỗ trợ kiến trúc:', supported)

# Thử thực sự 1 phép conv trên GPU để chắc chắn có CUDA kernel tương thích
try:
    _x = torch.randn(2, 3, 8, 8, device='cuda')
    _conv = torch.nn.Conv2d(3, 4, 3).cuda()
    _ = _conv(_x)
    torch.cuda.synchronize()
    print(f'✅ GPU {name} (sm_{cc}) TƯƠNG THÍCH — sẵn sàng huấn luyện.')
except RuntimeError as e:
    raise SystemExit(
        f'❌ GPU "{name}" (sm_{cc}) KHÔNG tương thích với PyTorch {torch.__version__}.\n'
        f'   PyTorch hiện chỉ hỗ trợ: {supported}.\n'
        f'   👉 Session options (⚙️ bên phải) → Accelerator → đổi sang "GPU T4 x2" → Save → Restart & Run All.\n'
        f'   (P100/sm_60 đã bị PyTorch >= 2.10 loại bỏ; T4 = sm_75 chạy tốt.)\n\n'
        f'Chi tiết lỗi: {e}'
    )


In [ ]:
# CELL 3 — Định vị & đồng bộ mã nguồn (dataset odir5k-code)
# Kaggle có thể lưu thư mục con dưới dạng .zip (--dir-mode zip) → xử lý cả 2 trường hợp.
import shutil, zipfile

CODE_INPUT = None
if os.path.isdir('/kaggle/input/odir5k-code'):
    CODE_INPUT = '/kaggle/input/odir5k-code'
else:
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'train.py' in files and ('src' in dirs or 'src.zip' in files):
            CODE_INPUT = root
            break
assert CODE_INPUT, '❌ Không tìm thấy dataset code (odir5k-code). Hãy Add Data.'
print('✅ Code dataset:', CODE_INPUT)

CODE_DIR = '/kaggle/working/code'
os.makedirs(CODE_DIR, exist_ok=True)
for name in ['src', 'configs', 'splits_clean']:
    dest = f'{CODE_DIR}/{name}'
    if os.path.exists(dest):
        shutil.rmtree(dest)
    src_dir = f'{CODE_INPUT}/{name}'
    src_zip = f'{CODE_INPUT}/{name}.zip'
    if os.path.isdir(src_dir):
        shutil.copytree(src_dir, dest)
    elif os.path.exists(src_zip):
        with zipfile.ZipFile(src_zip) as z:
            z.extractall(CODE_DIR)
for f in ['train.py', 'evaluate.py']:
    shutil.copy(f'{CODE_INPUT}/{f}', f'{CODE_DIR}/{f}')
sys.path.insert(0, CODE_DIR)
RES_DIR = '/kaggle/working/results'
os.makedirs(RES_DIR, exist_ok=True)

# Kiểm tra đồng bộ thành công
assert os.path.exists(f'{CODE_DIR}/src/config.py'), '❌ Đồng bộ src/ thất bại'
assert os.path.exists(f'{CODE_DIR}/splits_clean/train.csv'), '❌ Thiếu splits_clean/'
print('✅ Đã đồng bộ code vào', CODE_DIR)

In [ ]:
# CELL 4 — Định vị ảnh gốc ODIR-5K (Training Images)
RAW_DIR = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'Training Images' in dirs:
        cand = os.path.join(root, 'Training Images')
        if len([f for f in os.listdir(cand) if f.lower().endswith('.jpg')]) > 1000:
            RAW_DIR = cand
            break
assert RAW_DIR, '❌ Không tìm thấy ODIR-5K Training Images. Hãy Add Data ảnh gốc ODIR.'
print('✅ Ảnh gốc:', RAW_DIR, '|', len(os.listdir(RAW_DIR)), 'ảnh')

In [ ]:
# CELL 5 — Sinh ảnh ENHANCED (ROI Crop 512 → Ben Graham → CLAHE) vào /kaggle/working/enhanced_images
import cv2, numpy as np, pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

ENH_DIR = '/kaggle/working/enhanced_images'
os.makedirs(ENH_DIR, exist_ok=True)

def crop_roi(img, tol=7):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    rows, cols = np.where(gray > tol)
    if len(rows) == 0:
        return cv2.resize(img, (512, 512))
    r0, r1, c0, c1 = rows.min(), rows.max(), cols.min(), cols.max()
    return cv2.resize(img[r0:r1+1, c0:c1+1], (512, 512))

def ben_graham(img, sigma_ratio=1/6, scale=128):
    h, w = img.shape[:2]
    sigma = int(max(h, w) * sigma_ratio)
    if sigma % 2 == 0:
        sigma += 1
    local = cv2.GaussianBlur(img.astype(np.float32), (0, 0), sigmaX=sigma)
    return np.clip(img.astype(np.float32) - local + scale, 0, 255).astype(np.uint8)

def clahe_lab(img, clip=2.0, grid=(8, 8)):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=clip, tileGridSize=grid).apply(l)
    return cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)

# Chỉ xử lý các file có trong splits (đủ cho train/val/test)
SPLITS = f'{CODE_DIR}/splits_clean'
need = set()
for s in ['train', 'val', 'test']:
    need |= set(pd.read_csv(f'{SPLITS}/{s}.csv')['filename'].astype(str))

done = err = 0
for fn in tqdm(sorted(need), desc='Enhancing'):
    out = f'{ENH_DIR}/{fn}'
    if os.path.exists(out):
        done += 1; continue
    src = f'{RAW_DIR}/{fn}'
    img = cv2.imread(src)
    if img is None:
        err += 1; continue
    img = clahe_lab(ben_graham(crop_roi(img)))
    cv2.imwrite(out, img, [cv2.IMWRITE_JPEG_QUALITY, 95])
    done += 1
print(f'✅ Enhanced: {done} ảnh (lỗi: {err}) → {ENH_DIR}')

In [ ]:
# CELL 6 — Dry-run kiểm tra pipeline (1 batch)
cfg = f'{CODE_DIR}/configs/exp_6_swin_binary_enhanced_aug.yaml'
os.system(f'PYTHONPATH={CODE_DIR} python3 -u {CODE_DIR}/train.py --config {cfg} --dry-run')

In [ ]:
# CELL 7 — Huấn luyện các thực nghiệm
# Mỗi EXP lưu vào /kaggle/working/results/<exp>/
EXPERIMENTS = [
    'exp_4_swin_binary_raw',
    'exp_5_swin_binary_enhanced',
    'exp_6_swin_binary_enhanced_aug',
]
for exp in EXPERIMENTS:
    cfg = f'{CODE_DIR}/configs/{exp}.yaml'
    print(f'\n{"="*64}\n  TRAIN: {exp}\n{"="*64}')
    rc = os.system(f'PYTHONPATH={CODE_DIR} python3 -u {CODE_DIR}/train.py --config {cfg}')
    print(f'  → return code: {rc}')

In [ ]:
# CELL 8 — So sánh kết quả (bảng Ablation Study)
exps_arg = ' '.join(f'{RES_DIR}/{e}' for e in EXPERIMENTS)
os.system(f'PYTHONPATH={CODE_DIR} python3 {CODE_DIR}/evaluate.py --exps {exps_arg} --results-dir {RES_DIR}')
from IPython.display import Markdown, display
p = f'{RES_DIR}/comparison_table.md'
if os.path.exists(p):
    display(Markdown(open(p, encoding='utf-8').read()))

In [ ]:
# CELL 9 — Đường cong học tập (val AUC & val F1 theo epoch)
import pandas as pd, matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for exp in EXPERIMENTS:
    csv = f'{RES_DIR}/{exp}/training_log.csv'
    if not os.path.exists(csv):
        continue
    df = pd.read_csv(csv)
    axes[0].plot(df['epoch'], df['val_auc'], label=exp, marker='o', ms=3)
    axes[1].plot(df['epoch'], df['val_f1'], label=exp, marker='o', ms=3)
axes[0].set_title('Validation AUC-ROC'); axes[0].set_xlabel('Epoch'); axes[0].grid(alpha=.3); axes[0].legend(fontsize=7)
axes[1].set_title('Validation F1'); axes[1].set_xlabel('Epoch'); axes[1].grid(alpha=.3); axes[1].legend(fontsize=7)
plt.tight_layout(); plt.savefig(f'{RES_DIR}/learning_curves.png', dpi=120); plt.show()
print('✅ Lưu:', f'{RES_DIR}/learning_curves.png')